# Module 04.08 — Tags (`tag`)

> Part of the **Elasticsearch Snapshot Training** course.
> Run `00_setup.ipynb` first to start the Docker stack and load sample data.


In [ ]:
import sys, json, time
sys.path.insert(0, '..')
from helpers import *

es = get_client()

---
## 4.8 Tags (`tag`)

Tags are lightweight labels — a name, a colour, and an optional description — that you
can attach to any saved object for organisation and filtering in the Saved Objects UI and
in Kibana's global search bar.

What makes tags interesting from a schema perspective is the **reference direction**.
You might expect the tag to hold a list of objects it has been applied to, but that is
not how it works. Instead, *the tagged objects hold a reference pointing back to the tag*.
A dashboard with two tags will have two entries in its `references` array of type `tag`.
The tag object itself is tiny and has no references of its own.

This matters at restore time: restoring a tagged dashboard without its referenced tags
will succeed (Kibana tolerates missing tag references), but the tags won't appear in the
filter UI. A complete feature-state restore brings back both tags and their tagged objects
simultaneously, so the relationship is always intact.

Tags are global within a Kibana Space — any object in the space can reference any tag in
the same space. They are not shared across Spaces.

In [ ]:
heading('4.8 Tags — create and apply')

tag_resp = kibana_post('/api/saved_objects/tag', {
    'attributes': {
        'name': 'training',
        'description': 'Created by snapshot training module',
        'color': '#007bff',
    },
})
tag_id = tag_resp['id']
success(f'Tag created: id={tag_id}')

pp(tag_resp, 'tag saved object')
info('Key fields:')
info('  name        — display label')
info('  color       — hex color for the badge')
info('  description — tooltip text')
info('  Tags are referenced FROM tagged objects — the tag itself has no references array')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/management/kibana/tags', 'Stack Management → Tags')

In [ ]:
# Apply the tag to an existing dashboard (if any)
dashboards = find_saved_objects('dashboard')
if dashboards:
    d = dashboards[0]
    existing_refs = d.get('references', [])
    # Update the dashboard's references to include the tag
    new_refs = existing_refs + [{'type': 'tag', 'id': tag_id, 'name': f'tag-{tag_id}'}]
    kibana_post(f'/api/saved_objects/dashboard/{d["id"]}?overwrite=true', {
        'attributes': d['attributes'],
        'references': new_refs,
    })
    success(f"Tag applied to dashboard '{d['attributes']['title']}'")

    # Confirm the reference exists
    updated = kibana_get(f"/api/saved_objects/dashboard/{d['id']}")
    tag_refs = [r for r in updated.get('references', []) if r['type'] == 'tag']
    console.print(f'  Tag references on dashboard: {tag_refs}')

In [ ]:
heading('Tag — snapshot → delete → restore')

# Ensure tag exists (re-runnable)
existing_tag = next((o for o in find_saved_objects('tag') if o['attributes'].get('name') == 'training'), None)
if existing_tag:
    tag_id = existing_tag['id']
    info(f'Tag already exists: id={tag_id}')
else:
    tag_resp = kibana_post('/api/saved_objects/tag', {
        'attributes': {'name': 'training', 'description': 'Objects created during snapshot training', 'color': '#0077cc'},
    })
    tag_id = tag_resp['id']
    success(f'Tag recreated: id={tag_id}')

obj = kibana_get(f'/api/saved_objects/tag/{tag_id}')
pp(obj, 'tag saved object')
info('Key fields:')
info('  name        — display label')
info('  color       — hex colour shown in the UI badge')
info('  description — optional tooltip text')
info('  Note: tag assignment is stored in the TAGGED object\'s references, not the tag itself')

snap_delete_restore_cycle('snap-tag-test', 'Tag')

kibana_delete(f'/api/saved_objects/tag/{tag_id}')
warn(f'Accidentally deleted tag {tag_id}')
assert not any(o['id'] == tag_id for o in find_saved_objects('tag')), 'Should be gone'

restore_kibana_state(es, SNAP_REPO, 'snap-tag-test')
time.sleep(3)

restored = find_saved_objects('tag')
assert any(o['id'] == tag_id for o in restored), 'Tag should be restored'
success(f'Tag {tag_id} successfully restored!')
kibana_link('/app/management/kibana/tags', 'Stack Management → Kibana → Tags — verify tag is back')